In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from umap import UMAP

In [ ]:
df_2022 = pd.read_parquet("year_TERZ_parquets/TERZ_HHZ_hw6_ont25_2022.parquet")
df_2023 = pd.read_parquet("year_TERZ_parquets/TERZ_HHZ_hw6_ont25_2023.parquet")
df_2024 = pd.read_parquet("year_TERZ_parquets/TERZ_HHZ_hw6_ont25_2024.parquet")
df_2025 = pd.read_parquet("year_TERZ_parquets/TERZ_HHZ_hw6_ont25_2025.parquet")

df = pd.concat([df_2022, df_2023, df_2024, df_2025], ignore_index=True)
print(f"Combined dataset shape: {df.shape}")

In [ ]:
df.head(5)

In [ ]:
feature_columns = [col for col in df.columns if col.startswith("feature_")]
if len(feature_columns) == 0:
    raise ValueError("No feature_* columns found.")

X = df[feature_columns].to_numpy(dtype=np.float64)

print(f"Feature matrix: X.shape = {X.shape}")

In [ ]:
X_max = X.max(axis=1, keepdims=True)
X_norm = X / X_max
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_norm)
print(f"Scaled matrix: {X_scaled.shape}") 

\sqrt(64382) \approx 253.74 (we will use this value as the first attempt for perplexity)

In [ ]:
RANDOM_STATE = 42
METRIC = "euclidean"
PERPLEXITY = 255
EARLY_EXAGGERATION = 14

In [ ]:
CACHE_DIR = Path("TERZ_all_years_embeddings_cache")      
CACHE_DIR.mkdir(exist_ok=True)

In [ ]:
tsne = TSNE(n_components=2,perplexity=PERPLEXITY,early_exaggeration=EARLY_EXAGGERATION,init="pca",learning_rate="auto",random_state=RANDOM_STATE, metric=METRIC)
X_tsne = tsne.fit_transform(X_scaled)

In [ ]:
tag = f"tsne_perp{PERPLEXITY}_ee{EARLY_EXAGGERATION}"
CACHE_DIR / f"TERZ_hw6_ont25_all_years_{tag}.npy"

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1],s=5,alpha=0.5)
ax.set_title(f"tSNE | Perplexity = {PERPLEXITY} | Early Exaggeration = {EARLY_EXAGGERATION} | PCA Components = No PCA | Metric = {METRIC}")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")

plt.tight_layout()
plt.show()

In [ ]:
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1

In [ ]:
umap = UMAP(n_components=2, n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST, metric=METRIC, random_state=RANDOM_STATE)
X_umap = umap.fit_transform(X_scaled)

In [ ]:
tag = f"umap_nn{UMAP_N_NEIGHBORS}_md{UMAP_MIN_DIST}"
CACHE_DIR / f"TERZ_hw6_ont25_all_years_{tag}.npy"

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
scatter = ax.scatter(X_umap[:, 0], X_umap[:, 1],s=5,alpha=0.5)
ax.set_title(f"uamp | N_Neighbors = {UMAP_N_NEIGHBORS} | Min Dist. = {UMAP_MIN_DIST} | PCA Components = No PCA | Metric = {METRIC}")
ax.set_xlabel("umap 1")
ax.set_ylabel("umap 2")

plt.tight_layout()
plt.show()

## Density Analysis

In [ ]:
from scipy.stats import gaussian_kde
from scipy.ndimage import maximum_filter
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.manifold import TSNE

In [ ]:
# KDE Gaussian

def density_surface(E, grid):
    """Gaussian KDE of the 2D embedding evaluated on a regular grid."""
    grid = grid
    x, y = E[:, 0], E[:, 1]
    kde = gaussian_kde(np.vstack([x, y]))
    X, Y = np.meshgrid(np.linspace(x.min(), x.max(), grid), np.linspace(y.min(), y.max(), grid))
    Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
    return X, Y, Z

In [ ]:
X1, Y1, Z1 = density_surface(X_tsne, 100) #tSNE
X2, Y2, Z2 = density_surface(X_umap, 100) #umap

In [ ]:
fig = plt.figure(figsize=(18, 16))

ax = fig.add_subplot(2, 2, 1)
ax.scatter(X_tsne[:, 0], X_tsne[:, 1], s=5, alpha=0.5)

ax = fig.add_subplot(2, 2, 2, projection="3d")
surf = ax.plot_surface(X1, Y1, Z1, cmap="coolwarm", linewidth=0, antialiased=False)
ax.set_xlabel("tSNE-1")
ax.set_ylabel("tSNE-2")
ax.set_zlabel("density")
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.1)
ax.set_title(f"Gaussian KDE density on the tSNE embedding")

ax = fig.add_subplot(2, 2, 3)
ax.scatter(X_umap[:, 0], X_umap[:, 1], s=5, alpha=0.5)

ax = fig.add_subplot(2, 2, 4, projection="3d")
surf = ax.plot_surface(X2, Y2, Z2, cmap="coolwarm", linewidth=0, antialiased=False)
ax.set_xlabel("umap-1")
ax.set_ylabel("umap-2")
ax.set_zlabel("density")
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.1)
ax.set_title(f"Gaussian KDE density on the umap embedding")

plt.show()

In [ ]:
def kde_peaks(X, Y, Z, size, min_rel):
    """Local maxima of the KDE surface on the grid, tallest first -> ((k, 2) positions, (k,) heights)."""
    is_peak = (Z == maximum_filter(Z, size=size)) & (Z >= min_rel * Z.max())
    rows, cols = np.nonzero(is_peak)
    order = np.argsort(Z[rows, cols])[::-1]
    rows, cols = rows[order], cols[order]
    return np.column_stack([X[rows, cols], Y[rows, cols]]), Z[rows, cols]